In [52]:
import ast
import copy
import json
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional


ATTRIBUTE_TYPE_FOR_PY = {
    "str": "AttributeType.STRING",
    "int": "AttributeType.INTEGER",
    "float": "AttributeType.DOUBLE",
    "bool": "AttributeType.BOOLEAN",
}


@dataclass
class FunctionFacts:
    name: str
    params: list[str]
    returns_annotation: str | None
    called_functions: set[str] = field(default_factory=set)
    has_loop: bool = False
    recursive: bool = False
    family: str = "table"  # source | table | batch | tuple
    is_sink: bool = False
    globals_used: set[str] = field(default_factory=set)


@dataclass
class UiParamSpec:
    name: str
    attr_type: str
    default_ast: ast.AST | None = None


@dataclass
class DataInputSpec:
    param_name: str
    upstream_stage_id: str
    upstream_var: str
    input_port: int


@dataclass
class StageInvocation:
    stage_id: str
    function_name: str
    family: str
    is_source: bool
    is_sink: bool
    recursive: bool
    has_loop: bool
    assigned_to: list[str]
    data_inputs: list[DataInputSpec] = field(default_factory=list)
    ui_params: list[UiParamSpec] = field(default_factory=list)
    operator_id: str = ""
    display_name: str = ""
    code: str = ""
    operator_type: str = "PythonUDFV2"
    source_columns: list[dict[str, str]] = field(default_factory=list)
    output_columns: list[dict[str, str]] = field(default_factory=list)
    retain_input_columns: bool = True


class SourceModuleParser:
    def parse(self, source: str) -> ast.Module:
        return ast.parse(source)


class ImportCollector:
    def collect_nodes(self, tree: ast.Module) -> list[ast.stmt]:
        return [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))]

    def collect_names(self, tree: ast.Module) -> set[str]:
        names = set()
        for node in self.collect_nodes(tree):
            if isinstance(node, ast.Import):
                for alias in node.names:
                    names.add(alias.asname or alias.name.split(".")[0])
            else:
                for alias in node.names:
                    names.add(alias.asname or alias.name)
        return names


class FunctionIndex:
    def collect(self, tree: ast.Module) -> dict[str, ast.FunctionDef]:
        return {n.name: n for n in tree.body if isinstance(n, ast.FunctionDef)}


class CallCollector(ast.NodeVisitor):
    def __init__(self):
        self.calls: list[ast.Call] = []

    def visit_Call(self, node: ast.Call):
        self.calls.append(node)
        self.generic_visit(node)


class CallGraphBuilder:
    def build(self, functions: dict[str, ast.FunctionDef]) -> dict[str, set[str]]:
        graph = {name: set() for name in functions}
        for name, fn in functions.items():
            collector = CallCollector()
            collector.visit(fn)
            for call in collector.calls:
                if isinstance(call.func, ast.Name) and call.func.id in functions:
                    graph[name].add(call.func.id)
        return graph


class RecursiveAnalyzer:
    def strongly_connected_components(self, graph: dict[str, set[str]]) -> list[list[str]]:
        index = 0
        stack: list[str] = []
        on_stack: set[str] = set()
        indices: dict[str, int] = {}
        lowlink: dict[str, int] = {}
        comps: list[list[str]] = []

        def strongconnect(v: str):
            nonlocal index
            indices[v] = index
            lowlink[v] = index
            index += 1
            stack.append(v)
            on_stack.add(v)

            for w in graph[v]:
                if w not in indices:
                    strongconnect(w)
                    lowlink[v] = min(lowlink[v], lowlink[w])
                elif w in on_stack:
                    lowlink[v] = min(lowlink[v], indices[w])

            if lowlink[v] == indices[v]:
                comp = []
                while True:
                    w = stack.pop()
                    on_stack.remove(w)
                    comp.append(w)
                    if w == v:
                        break
                comps.append(comp)

        for v in graph:
            if v not in indices:
                strongconnect(v)
        return comps

    def detect(self, graph: dict[str, set[str]]) -> set[str]:
        recursive = set()
        for comp in self.strongly_connected_components(graph):
            if len(comp) > 1:
                recursive.update(comp)
            elif len(comp) == 1 and comp[0] in graph[comp[0]]:
                recursive.add(comp[0])
        return recursive


class NameUsageCollector(ast.NodeVisitor):
    def __init__(self):
        self.loads: set[str] = set()
        self.stores: set[str] = set()

    def visit_Name(self, node: ast.Name):
        if isinstance(node.ctx, ast.Load):
            self.loads.add(node.id)
        elif isinstance(node.ctx, ast.Store):
            self.stores.add(node.id)

    def visit_FunctionDef(self, node):
        return

    def visit_ClassDef(self, node):
        return


class FamilyInferer:
    def infer(self, fn: ast.FunctionDef) -> str:
        ann_texts = []
        for a in fn.args.args:
            if a.annotation is not None:
                ann_texts.append(ast.unparse(a.annotation))
        if fn.returns is not None:
            ann_texts.append(ast.unparse(fn.returns))
        joined = " | ".join(ann_texts)

        if any(tok in joined for tok in ["TupleLike", "Tuple", "tuple_"]):
            return "tuple"
        if any(tok in joined for tok in ["BatchLike", "Batch"]):
            return "batch"
        if any(tok in joined for tok in ["DataFrame", "TableLike", "Table", "pd.DataFrame"]):
            return "table"

        source = ast.unparse(fn)
        if any(tok in source for tok in ["pd.", ".copy(", ".dropna(", ".sort_values(", ".rolling("]):
            return "table"
        return "table"


class SinkDetector:
    def detect(self, fn: ast.FunctionDef) -> bool:
        source = ast.unparse(fn)
        if any(tok in source for tok in ["to_csv(", ".show(", "plt.", ".save(", "print("]):
            return True
        if fn.returns is None:
            for stmt in fn.body:
                if isinstance(stmt, ast.Return) and stmt.value is not None:
                    return False
            return True
        return False


class LoopCollector(ast.NodeVisitor):
    def __init__(self, function_name: str):
        self.function_name = function_name
        self.has_loop = False

    def visit_For(self, node: ast.For):
        self.has_loop = True
        self.generic_visit(node)

    def visit_While(self, node: ast.While):
        self.has_loop = True
        self.generic_visit(node)


class FunctionFactsAnalyzer:
    def __init__(self):
        self.family_inferer = FamilyInferer()
        self.sink_detector = SinkDetector()

    def analyze(
        self,
        tree: ast.Module,
        functions: dict[str, ast.FunctionDef],
        graph: dict[str, set[str]],
        recursive: set[str],
    ) -> dict[str, FunctionFacts]:
        import_names = ImportCollector().collect_names(tree)
        facts: dict[str, FunctionFacts] = {}
        for name, fn in functions.items():
            usage = NameUsageCollector()
            usage.visit(fn)
            loops = LoopCollector(name)
            loops.visit(fn)
            facts[name] = FunctionFacts(
                name=name,
                params=[a.arg for a in fn.args.args],
                returns_annotation=ast.unparse(fn.returns) if fn.returns is not None else None,
                called_functions=set(graph[name]),
                has_loop=loops.has_loop,
                recursive=name in recursive,
                family=self.family_inferer.infer(fn),
                is_sink=self.sink_detector.detect(fn),
                globals_used=(usage.loads - usage.stores - {a.arg for a in fn.args.args}) & import_names,
            )
        return facts


class SourceSchemaInferer:
    def __init__(self):
        try:
            import pandas as _pd
        except Exception:
            _pd = None
        self.pd = _pd

    def _literal_value(self, node: ast.AST | None):
        if isinstance(node, ast.Constant):
            return node.value
        return None

    def _resolve_expr(self, expr: ast.AST | None, ui_defaults: dict[str, Any]):
        if expr is None:
            return None
        if isinstance(expr, ast.Constant):
            return expr.value
        if isinstance(expr, ast.Name):
            return ui_defaults.get(expr.id)
        return None

    def _map_dtype(self, dtype: Any) -> str:
        kind = getattr(dtype, "kind", None)
        if kind == "b":
            return "boolean"
        if kind == "i":
            try:
                itemsize = int(getattr(dtype, "itemsize", 8) or 8)
            except Exception:
                itemsize = 8
            return "integer" if itemsize <= 4 else "long"
        if kind == "u":
            try:
                itemsize = int(getattr(dtype, "itemsize", 8) or 8)
            except Exception:
                itemsize = 8
            return "integer" if itemsize <= 4 else "long"
        if kind == "f":
            return "double"
        if kind == "M":
            return "timestamp"
        if kind == "S":
            return "binary"
        return "string"

    def infer(self, fn: ast.FunctionDef, stage: StageInvocation) -> list[dict[str, str]]:
        if not stage.is_source or stage.family != "table" or self.pd is None:
            return []

        ui_defaults = {spec.name: self._literal_value(spec.default_ast) for spec in stage.ui_params}

        for node in ast.walk(fn):
            if not isinstance(node, ast.Call):
                continue
            if not isinstance(node.func, ast.Attribute):
                continue
            if not (isinstance(node.func.value, ast.Name) and node.func.value.id == "pd" and node.func.attr == "read_csv"):
                continue

            source_expr = None
            if node.args:
                source_expr = node.args[0]
            else:
                for kw in node.keywords:
                    if kw.arg in ("filepath_or_buffer", "path", "url"):
                        source_expr = kw.value
                        break

            source_value = self._resolve_expr(source_expr, ui_defaults)
            if not isinstance(source_value, str) or not source_value:
                continue

            try:
                df = self.pd.read_csv(source_value, nrows=50)
            except Exception:
                continue

            cols: list[dict[str, str]] = []
            for col in df.columns:
                try:
                    dtype = df[col].dtype
                except Exception:
                    dtype = None
                cols.append({
                    "attributeName": str(col),
                    "attributeType": self._map_dtype(dtype),
                })
            if cols:
                return cols

        return []


class TableOutputSchemaInferer:
    def _schema_dict(self, cols: list[dict[str, str]]) -> dict[str, str]:
        return {c["attributeName"]: c["attributeType"] for c in cols if c.get("attributeName")}

    def _schema_list(self, d: dict[str, str]) -> list[dict[str, str]]:
        return [{"attributeName": k, "attributeType": v} for k, v in d.items()]

    def _list_of_constant_strings(self, node: ast.AST | None) -> list[str] | None:
        if isinstance(node, (ast.List, ast.Tuple)):
            vals = []
            for elt in node.elts:
                if isinstance(elt, ast.Constant) and isinstance(elt.value, str):
                    vals.append(elt.value)
                else:
                    return None
            return vals
        return None

    def _infer_scalar_type(self, expr: ast.AST, env: dict[str, dict[str, str]]) -> str:
        if isinstance(expr, ast.Constant):
            v = expr.value
            if isinstance(v, bool):
                return "boolean"
            if isinstance(v, int):
                return "long"
            if isinstance(v, float):
                return "double"
            return "string"
        if isinstance(expr, ast.Subscript) and isinstance(expr.value, ast.Name):
            if isinstance(expr.slice, ast.Constant) and isinstance(expr.slice.value, str):
                return env.get(expr.value.id, {}).get(expr.slice.value, "string")
        if isinstance(expr, ast.BinOp):
            return "double"
        if isinstance(expr, ast.Call) and isinstance(expr.func, ast.Attribute):
            attr = expr.func.attr
            if isinstance(expr.func.value, ast.Name) and expr.func.value.id == "pd" and attr == "to_datetime":
                return "timestamp"
            if attr in {"pct_change", "mean", "sum", "std", "median", "max", "min", "rolling"}:
                return "double"
        return "string"

    def _resolve_table_schema(self, expr: ast.AST, env: dict[str, dict[str, str]]) -> dict[str, str] | None:
        if isinstance(expr, ast.Name):
            schema = env.get(expr.id)
            return dict(schema) if schema is not None else None
        if isinstance(expr, ast.Subscript):
            base = self._resolve_table_schema(expr.value, env)
            cols = self._list_of_constant_strings(expr.slice)
            if base is not None and cols is not None:
                return {c: base[c] for c in cols if c in base}
            return None
        if isinstance(expr, ast.Call) and isinstance(expr.func, ast.Attribute):
            base = self._resolve_table_schema(expr.func.value, env)
            attr = expr.func.attr
            if base is None:
                return None
            if attr in {"copy", "dropna", "sort_values", "reset_index", "rename_axis", "fillna", "head", "tail"}:
                return dict(base)
            if attr == "assign":
                out = dict(base)
                for kw in expr.keywords:
                    if kw.arg:
                        out[kw.arg] = self._infer_scalar_type(kw.value, env)
                return out
            return dict(base)
        return None

    def infer(self, fn: ast.FunctionDef, stage: StageInvocation, input_columns: list[dict[str, str]]) -> tuple[list[dict[str, str]], bool]:
        if stage.family != "table":
            return [], True
        env: dict[str, dict[str, str]] = {}
        if stage.data_inputs:
            if len(stage.data_inputs) == 1:
                env[stage.data_inputs[0].param_name] = self._schema_dict(input_columns)
            else:
                merged: dict[str, str] = {}
                for spec in stage.data_inputs:
                    for c in input_columns:
                        if c.get("attributeName"):
                            merged[c["attributeName"]] = c["attributeType"]
                    env[spec.param_name] = dict(merged)
        elif stage.is_source and stage.source_columns:
            # Seed read_csv-assigned variables lazily below.
            pass

        return_schema: dict[str, str] | None = None
        for stmt in fn.body:
            if isinstance(stmt, ast.Assign):
                value_schema = self._resolve_table_schema(stmt.value, env)
                if value_schema is not None:
                    for t in stmt.targets:
                        if isinstance(t, ast.Name):
                            env[t.id] = dict(value_schema)
                for t in stmt.targets:
                    if isinstance(t, ast.Subscript) and isinstance(t.value, ast.Name):
                        root = t.value.id
                        if root not in env:
                            continue
                        if isinstance(t.slice, ast.Constant) and isinstance(t.slice.value, str):
                            env[root][t.slice.value] = self._infer_scalar_type(stmt.value, env)
            elif isinstance(stmt, ast.Return) and stmt.value is not None:
                rs = self._resolve_table_schema(stmt.value, env)
                if rs is not None:
                    return_schema = rs

        if return_schema is None:
            if stage.is_source:
                return_schema = self._schema_dict(stage.source_columns)
            elif len(stage.data_inputs) == 1:
                return_schema = self._schema_dict(input_columns)
            else:
                return_schema = {}
                for c in input_columns:
                    if c.get("attributeName"):
                        return_schema[c["attributeName"]] = c["attributeType"]

        input_schema_dict = self._schema_dict(input_columns)
        retain = return_schema == input_schema_dict
        # Emit explicit output columns for all table stages to keep runtime schema aligned.
        return self._schema_list(return_schema), False if return_schema else retain


class WorkflowIdFactory:
    def operator_id(self, operator_type: str = "PythonUDFV2") -> str:
        return f"{operator_type}-operator-{uuid.uuid4()}"

    def link_id(self) -> str:
        return f"link-{uuid.uuid4()}"

    def stage_id(self, base: str, idx: int) -> str:
        return f"{base}__{idx}"


class ConstantEnvBuilder:
    def build(self, entry_fn: ast.FunctionDef) -> dict[str, ast.AST]:
        env: dict[str, ast.AST] = {}
        for stmt in entry_fn.body:
            if isinstance(stmt, ast.Assign) and isinstance(stmt.value, ast.Constant):
                for t in stmt.targets:
                    if isinstance(t, ast.Name):
                        env[t.id] = copy.deepcopy(stmt.value)
        return env


class AttributeTypeInferer:
    def from_annotation(self, annotation: ast.AST | None) -> str:
        if annotation is None:
            return "AttributeType.STRING"
        text = ast.unparse(annotation)
        if text in ("str", "Optional[str]"):
            return "AttributeType.STRING"
        if text in ("int", "Optional[int]"):
            return "AttributeType.INTEGER"
        if text in ("float", "Optional[float]"):
            return "AttributeType.DOUBLE"
        if text in ("bool", "Optional[bool]"):
            return "AttributeType.BOOLEAN"
        return "AttributeType.STRING"

    def from_value(self, node: ast.AST | None) -> str:
        if isinstance(node, ast.Constant):
            py_name = type(node.value).__name__
            return ATTRIBUTE_TYPE_FOR_PY.get(py_name, "AttributeType.STRING")
        return "AttributeType.STRING"


class EntryDagExtractor:
    def __init__(self):
        self.type_inferer = AttributeTypeInferer()
        self.ids = WorkflowIdFactory()

    def _assigned_names(self, stmt: ast.stmt) -> list[str]:
        names: list[str] = []
        if isinstance(stmt, ast.Assign):
            for t in stmt.targets:
                if isinstance(t, ast.Name):
                    names.append(t.id)
                elif isinstance(t, (ast.Tuple, ast.List)):
                    for elt in t.elts:
                        if isinstance(elt, ast.Name):
                            names.append(elt.id)
        return names

    def extract(
        self,
        entry_fn: ast.FunctionDef,
        functions: dict[str, ast.FunctionDef],
        facts: dict[str, FunctionFacts],
        constant_env: dict[str, ast.AST],
    ) -> list[StageInvocation]:
        stages: list[StageInvocation] = []
        producer_for_var: dict[str, str] = {}
        stage_index = 0

        for stmt in entry_fn.body:
            call = None
            if isinstance(stmt, ast.Assign) and isinstance(stmt.value, ast.Call):
                call = stmt.value
            elif isinstance(stmt, ast.Expr) and isinstance(stmt.value, ast.Call):
                call = stmt.value
            if not call or not isinstance(call.func, ast.Name) or call.func.id not in functions or call.func.id == entry_fn.name:
                continue

            fn_name = call.func.id
            fn = functions[fn_name]
            fact = facts[fn_name]
            assigned_to = self._assigned_names(stmt)
            data_inputs: list[DataInputSpec] = []
            ui_params: list[UiParamSpec] = []

            for i, param in enumerate(fn.args.args):
                arg_node = None
                if i < len(call.args):
                    arg_node = call.args[i]
                else:
                    # support exact-name keyword arguments
                    for kw in call.keywords:
                        if kw.arg == param.arg:
                            arg_node = kw.value
                            break

                if arg_node is None:
                    continue

                if isinstance(arg_node, ast.Name) and arg_node.id in producer_for_var:
                    data_inputs.append(DataInputSpec(
                        param_name=param.arg,
                        upstream_stage_id=producer_for_var[arg_node.id],
                        upstream_var=arg_node.id,
                        input_port=len(data_inputs),
                    ))
                else:
                    default_ast = copy.deepcopy(constant_env[arg_node.id]) if isinstance(arg_node, ast.Name) and arg_node.id in constant_env else copy.deepcopy(arg_node)
                    attr_type = self.type_inferer.from_value(default_ast)
                    if attr_type == "AttributeType.STRING":
                        attr_type = self.type_inferer.from_annotation(param.annotation) or attr_type
                    ui_params.append(UiParamSpec(name=param.arg, attr_type=attr_type, default_ast=default_ast))

            is_source = len(data_inputs) == 0
            stage = StageInvocation(
                stage_id=self.ids.stage_id(fn_name, stage_index),
                function_name=fn_name,
                family=fact.family,
                is_source=is_source,
                is_sink=fact.is_sink,
                recursive=fact.recursive,
                has_loop=fact.has_loop,
                assigned_to=assigned_to,
                data_inputs=data_inputs,
                ui_params=ui_params,
            )
            stages.append(stage)
            for name in assigned_to:
                producer_for_var[name] = stage.stage_id
            stage_index += 1

        return stages


class ParamRewriter(ast.NodeTransformer):
    def __init__(self, name_mapping: dict[str, ast.AST]):
        self.name_mapping = name_mapping

    def visit_Name(self, node: ast.Name):
        if isinstance(node.ctx, ast.Load) and node.id in self.name_mapping:
            return ast.copy_location(copy.deepcopy(self.name_mapping[node.id]), node)
        return node


class YieldTransformer:
    def __init__(self, family: str, is_source: bool, is_sink: bool, passthrough_name: str):
        self.family = family
        self.is_source = is_source
        self.is_sink = is_sink
        self.passthrough_name = passthrough_name

    def _yield_stmt(self, value: ast.AST | None) -> ast.stmt:
        return ast.Expr(value=ast.Yield(value=value))

    def transform_block(self, body: list[ast.stmt], top_level: bool = False) -> list[ast.stmt]:
        new_body: list[ast.stmt] = []
        for stmt in body:
            if isinstance(stmt, ast.Return):
                if stmt.value is not None:
                    new_body.append(self._yield_stmt(stmt.value))
                elif top_level and (not self.is_source) and self.is_sink:
                    new_body.append(self._yield_stmt(ast.Name(id=self.passthrough_name, ctx=ast.Load())))
                new_body.append(ast.Return(value=None))
            elif isinstance(stmt, ast.If):
                stmt.body = self.transform_block(stmt.body, top_level=False)
                stmt.orelse = self.transform_block(stmt.orelse, top_level=False)
                new_body.append(stmt)
            elif isinstance(stmt, ast.For):
                stmt.body = self.transform_block(stmt.body, top_level=False)
                stmt.orelse = self.transform_block(stmt.orelse, top_level=False)
                new_body.append(stmt)
            elif isinstance(stmt, ast.While):
                stmt.body = self.transform_block(stmt.body, top_level=False)
                stmt.orelse = self.transform_block(stmt.orelse, top_level=False)
                new_body.append(stmt)
            else:
                new_body.append(stmt)

        has_yield = any(isinstance(n, ast.Yield) for n in ast.walk(ast.Module(body=new_body, type_ignores=[])))
        if top_level and not has_yield and not self.is_source:
            new_body.append(self._yield_stmt(ast.Name(id=self.passthrough_name, ctx=ast.Load())))
        return new_body


class UdfBaseSelector:
    def select(self, family: str, is_source: bool) -> tuple[str, str, str]:
        if is_source:
            return "ProcessSourceOperator", "UDFSourceOperator", "produce"
        if family == "tuple":
            return "ProcessTupleOperator", "UDFOperatorV2", "process_tuple"
        if family == "batch":
            return "ProcessBatchOperator", "UDFBatchOperator", "process_batch"
        return "ProcessTableOperator", "UDFTableOperator", "process_table"

    def workflow_operator_type(self, family: str, is_source: bool, input_count: int) -> str:
        """
        Workflow JSON only uses the Python operator types that are known to exist
        in the target Texera deployment:
          - PythonUDFSourceV2 for 0-input source operators
          - PythonUDFV2 for regular 1-input Python UDFs
          - DualInputPortsPythonUDFV2 for 2-input Python UDFs

        The implementation class inside `code` may still be UDFOperatorV2,
        UDFBatchOperator, UDFTableOperator, or UDFSourceOperator.
        """
        if is_source:
            return "PythonUDFSourceV2"
        if input_count == 2:
            return "DualInputPortsPythonUDFV2"
        return "PythonUDFV2"

    def data_variable(self, family: str) -> str:
        return {"tuple": "tuple_", "batch": "batch"}.get(family, "table")

    def process_signature(self, family: str, is_source: bool) -> tuple[list[ast.arg], Optional[ast.AST]]:
        if is_source:
            return (
                [ast.arg(arg="self")],
                ast.Subscript(
                    value=ast.Name(id="Iterator", ctx=ast.Load()),
                    slice=ast.Subscript(
                        value=ast.Name(id="Optional", ctx=ast.Load()),
                        slice=ast.Subscript(
                            value=ast.Name(id="Union", ctx=ast.Load()),
                            slice=ast.Tuple(elts=[ast.Name(id="TupleLike", ctx=ast.Load()), ast.Name(id="TableLike", ctx=ast.Load())], ctx=ast.Load()),
                            ctx=ast.Load(),
                        ),
                        ctx=ast.Load(),
                    ),
                    ctx=ast.Load(),
                ),
            )
        if family == "tuple":
            return (
                [ast.arg(arg="self"), ast.arg(arg="tuple_", annotation=ast.Name(id="Tuple", ctx=ast.Load())), ast.arg(arg="port", annotation=ast.Name(id="int", ctx=ast.Load()))],
                ast.Subscript(value=ast.Name(id="Iterator", ctx=ast.Load()), slice=ast.Subscript(value=ast.Name(id="Optional", ctx=ast.Load()), slice=ast.Name(id="TupleLike", ctx=ast.Load()), ctx=ast.Load()), ctx=ast.Load()),
            )
        if family == "batch":
            return (
                [ast.arg(arg="self"), ast.arg(arg="batch", annotation=ast.Name(id="Batch", ctx=ast.Load())), ast.arg(arg="port", annotation=ast.Name(id="int", ctx=ast.Load()))],
                ast.Subscript(value=ast.Name(id="Iterator", ctx=ast.Load()), slice=ast.Subscript(value=ast.Name(id="Optional", ctx=ast.Load()), slice=ast.Name(id="BatchLike", ctx=ast.Load()), ctx=ast.Load()), ctx=ast.Load()),
            )
        return (
            [ast.arg(arg="self"), ast.arg(arg="table", annotation=ast.Name(id="Table", ctx=ast.Load())), ast.arg(arg="port", annotation=ast.Name(id="int", ctx=ast.Load()))],
            ast.Subscript(value=ast.Name(id="Iterator", ctx=ast.Load()), slice=ast.Subscript(value=ast.Name(id="Optional", ctx=ast.Load()), slice=ast.Name(id="TableLike", ctx=ast.Load()), ctx=ast.Load()), ctx=ast.Load()),
        )


class UiParameterEmitter:
    def statements(self, spec: UiParamSpec) -> list[ast.stmt]:
        stmts: list[ast.stmt] = []
        if spec.default_ast is not None:
            stmts.append(
                ast.Assign(
                    targets=[ast.Attribute(value=ast.Name(id="self", ctx=ast.Load()), attr=spec.name, ctx=ast.Store())],
                    value=copy.deepcopy(spec.default_ast),
                )
            )
        else:
            stmts.append(
                ast.Assign(
                    targets=[ast.Attribute(value=ast.Name(id="self", ctx=ast.Load()), attr=spec.name, ctx=ast.Store())],
                    value=ast.Constant(value=None),
                )
            )
        return stmts

    def commented_line(self, spec: UiParamSpec) -> str:
        attr_suffix = spec.attr_type.split(".")[-1]
        return f"# self.UiParameter({spec.name!r}, AttributeType.{attr_suffix})  # UiParameter disabled for now"

    def hardcoded_comment_line(self, spec: UiParamSpec) -> str:
        return f"# Temporary hardcoded default for UI parameter '{spec.name}' from the source script."


class CodeCommentInjector:
    def inject_ui_parameter_comments(self, code: str, stage: StageInvocation, ui_emitter: UiParameterEmitter) -> str:
        if not stage.ui_params:
            return code
        lines = code.splitlines()
        out: list[str] = []
        specs_by_line = {
            f"self.{spec.name} = {ast.unparse(spec.default_ast) if spec.default_ast is not None else 'None'}": spec
            for spec in stage.ui_params
        }
        for line in lines:
            out.append(line)
            stripped = line.strip()
            spec = specs_by_line.get(stripped)
            if spec is not None:
                indent = line[: len(line) - len(line.lstrip())]
                out.append(indent + ui_emitter.hardcoded_comment_line(spec))
                out.append(indent + ui_emitter.commented_line(spec))
        return "\n".join(out) + "\n"


class FunctionBodyCompiler:
    def __init__(self):
        self.base_selector = UdfBaseSelector()
        self.ui_emitter = UiParameterEmitter()
        self.comment_injector = CodeCommentInjector()

    def _param_mapping(self, stage: StageInvocation) -> dict[str, ast.AST]:
        mapping: dict[str, ast.AST] = {}
        for spec in stage.ui_params:
            mapping[spec.name] = ast.Attribute(value=ast.Name(id="self", ctx=ast.Load()), attr=spec.name, ctx=ast.Load())
        if stage.is_source:
            return mapping
        if len(stage.data_inputs) == 1:
            only = stage.data_inputs[0]
            mapping[only.param_name] = ast.Name(id=self.base_selector.data_variable(stage.family), ctx=ast.Load())
        elif len(stage.data_inputs) > 1:
            for spec in stage.data_inputs:
                mapping[spec.param_name] = ast.Name(id=f"input_{spec.input_port}", ctx=ast.Load())
        return mapping

    def _prep_statements(self, stage: StageInvocation) -> list[ast.stmt]:
        stmts: list[ast.stmt] = []
        if len(stage.data_inputs) > 1:
            data_var = self.base_selector.data_variable(stage.family)
            stmts.append(ast.Assign(
                targets=[ast.Subscript(value=ast.Attribute(value=ast.Name(id="self", ctx=ast.Load()), attr="_texera_inputs", ctx=ast.Load()), slice=ast.Name(id="port", ctx=ast.Load()), ctx=ast.Store())],
                value=ast.Name(id=data_var, ctx=ast.Load()),
            ))
            cond = ast.Compare(
                left=ast.Call(func=ast.Name(id="len", ctx=ast.Load()), args=[ast.Attribute(value=ast.Name(id="self", ctx=ast.Load()), attr="_texera_inputs", ctx=ast.Load())], keywords=[]),
                ops=[ast.Lt()],
                comparators=[ast.Constant(value=len(stage.data_inputs))],
            )
            stmts.append(ast.If(test=cond, body=[ast.Return(value=None)], orelse=[]))
            for spec in stage.data_inputs:
                stmts.append(ast.Assign(
                    targets=[ast.Name(id=f"input_{spec.input_port}", ctx=ast.Store())],
                    value=ast.Subscript(value=ast.Attribute(value=ast.Name(id="self", ctx=ast.Load()), attr="_texera_inputs", ctx=ast.Load()), slice=ast.Constant(value=spec.input_port), ctx=ast.Load()),
                ))
        return stmts

    def compile_body(self, fn: ast.FunctionDef, stage: StageInvocation) -> list[ast.stmt]:
        fn_copy = copy.deepcopy(fn)
        mapping = self._param_mapping(stage)
        fn_copy = ParamRewriter(mapping).visit(fn_copy)
        ast.fix_missing_locations(fn_copy)

        body: list[ast.stmt] = self._prep_statements(stage)
        for stmt in fn_copy.body:
            if isinstance(stmt, ast.Expr) and isinstance(stmt.value, ast.Constant) and isinstance(stmt.value.value, str):
                continue
            body.append(stmt)
        body = YieldTransformer(
            family=stage.family,
            is_source=stage.is_source,
            is_sink=stage.is_sink,
            passthrough_name=self.base_selector.data_variable(stage.family),
        ).transform_block(body, top_level=True)
        if not body:
            body = [ast.Pass()]
        return body

    def emit_class(self, import_nodes: list[ast.stmt], fn: ast.FunctionDef, stage: StageInvocation) -> str:
        class_name, base_name, method_name = self.base_selector.select(stage.family, stage.is_source)
        args_list, returns_ann = self.base_selector.process_signature(stage.family, stage.is_source)
        body = self.compile_body(fn, stage)

        open_body: list[ast.stmt] = []
        for spec in stage.ui_params:
            open_body.extend(self.ui_emitter.statements(spec))
        if len(stage.data_inputs) > 1:
            open_body.append(ast.Assign(
                targets=[ast.Attribute(value=ast.Name(id="self", ctx=ast.Load()), attr="_texera_inputs", ctx=ast.Store())],
                value=ast.Dict(keys=[], values=[]),
            ))
        if not open_body:
            open_body = [ast.Pass()]

        class_body: list[ast.stmt] = []
        if stage.recursive:
            class_body.append(ast.Expr(value=ast.Constant(value=f"TRANSLATOR NOTE: {stage.function_name} is recursive.")))
        if stage.has_loop:
            class_body.append(ast.Expr(value=ast.Constant(value=f"TRANSLATOR NOTE: {stage.function_name} contains loops.")))

        class_body.append(ast.FunctionDef(
            name="open",
            args=ast.arguments(posonlyargs=[], args=[ast.arg(arg="self")], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]),
            body=open_body,
            decorator_list=[],
            returns=ast.Name(id="None", ctx=ast.Load()),
            type_comment=None,
        ))
        class_body.append(ast.FunctionDef(
            name=method_name,
            args=ast.arguments(posonlyargs=[], args=args_list, vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]),
            body=body,
            decorator_list=[ast.Name(id="overrides", ctx=ast.Load())],
            returns=returns_ann,
            type_comment=None,
        ))
        class_body.append(ast.FunctionDef(
            name="close",
            args=ast.arguments(posonlyargs=[], args=[ast.arg(arg="self")], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]),
            body=[ast.Pass()],
            decorator_list=[],
            returns=ast.Name(id="None", ctx=ast.Load()),
            type_comment=None,
        ))
        cls = ast.ClassDef(name=class_name, bases=[ast.Name(id=base_name, ctx=ast.Load())], keywords=[], body=class_body, decorator_list=[])
        ast.fix_missing_locations(cls)

        prelude = [
            "from typing import Iterator, Optional, Union",
            *[ast.unparse(n) for n in import_nodes],
            "from pytexera import *",
            "from core.models.schema.attribute_type import AttributeType",
            "",
        ]
        module = ast.Module(body=[cls], type_ignores=[])
        ast.fix_missing_locations(module)
        rendered = "\n".join(prelude) + ast.unparse(module) + "\n"
        return self.comment_injector.inject_ui_parameter_comments(rendered, stage, self.ui_emitter)


class WorkflowOperatorBuilder:
    def _operator_properties(self, stage: StageInvocation) -> dict[str, Any]:
        props: dict[str, Any] = {
            "code": stage.code,
            "workers": 1,
        }
        if stage.operator_type == "PythonUDFSourceV2":
            if stage.source_columns:
                props["columns"] = stage.source_columns
        else:
            props["retainInputColumns"] = stage.retain_input_columns
            if stage.output_columns:
                props["outputColumns"] = stage.output_columns
        return props

    def _input_ports(self, stage: StageInvocation) -> list[dict[str, Any]]:
        if stage.is_source:
            return []
        count = max(1, len(stage.data_inputs))
        ports = []
        for i in range(count):
            display_name = ""
            if count > 1 and i < len(stage.data_inputs):
                display_name = stage.data_inputs[i].param_name
            dependencies: list[dict[str, Any]] = []
            if count > 1 and i > 0:
                dependencies = [{"id": i - 1, "internal": False}]
            ports.append({
                "portID": f"input-{i}",
                "displayName": display_name,
                "allowMultiInputs": True,
                "isDynamicPort": False,
                "dependencies": dependencies,
            })
        return ports

    def _dynamic_ports_flags(self, stage: StageInvocation) -> tuple[bool, bool]:
        if stage.is_source:
            return False, False
        if stage.operator_type == "DualInputPortsPythonUDFV2":
            return False, False
        return True, True

    def build(self, stage: StageInvocation, view_result: bool = False) -> dict[str, Any]:
        dynamic_input_ports, dynamic_output_ports = self._dynamic_ports_flags(stage)
        operator = {
            "operatorID": stage.operator_id,
            "operatorType": stage.operator_type,
            "operatorVersion": "N/A",
            "operatorProperties": self._operator_properties(stage),
            "inputPorts": self._input_ports(stage),
            "outputPorts": [
                {
                    "portID": "output-0",
                    "displayName": "",
                    "allowMultiInputs": False,
                    "isDynamicPort": False,
                }
            ],
            "showAdvanced": False,
            "isDisabled": False,
            "customDisplayName": stage.display_name,
            "dynamicInputPorts": dynamic_input_ports,
            "dynamicOutputPorts": dynamic_output_ports,
        }
        if view_result:
            operator["viewResult"] = True
        return operator


class WorkflowLinkBuilder:
    def __init__(self):
        self.ids = WorkflowIdFactory()

    def build_links(self, stages: list[StageInvocation]) -> list[dict[str, Any]]:
        stage_by_id = {s.stage_id: s for s in stages}
        links: list[dict[str, Any]] = []
        for stage in stages:
            for inp in stage.data_inputs:
                upstream = stage_by_id[inp.upstream_stage_id]
                links.append({
                    "linkID": self.ids.link_id(),
                    "source": {"operatorID": upstream.operator_id, "portID": "output-0"},
                    "target": {"operatorID": stage.operator_id, "portID": f"input-{inp.input_port}"},
                })
        return links


class WorkflowLayoutBuilder:
    def build_positions(self, stages: list[StageInvocation], start_x: int = 240, start_y: int = 180, x_gap: int = 300, y_gap: int = 180) -> dict[str, dict[str, int]]:
        preds: dict[str, list[str]] = {s.stage_id: [inp.upstream_stage_id for inp in s.data_inputs] for s in stages}
        layers: dict[str, int] = {}
        remaining = {s.stage_id for s in stages}
        while remaining:
            progressed = False
            for sid in list(remaining):
                if all(p in layers for p in preds[sid]):
                    layers[sid] = 0 if not preds[sid] else max(layers[p] for p in preds[sid]) + 1
                    remaining.remove(sid)
                    progressed = True
            if not progressed:
                for sid in list(remaining):
                    layers[sid] = 0
                    remaining.remove(sid)
        by_layer: dict[int, list[StageInvocation]] = {}
        for s in stages:
            by_layer.setdefault(layers[s.stage_id], []).append(s)
        positions: dict[str, dict[str, int]] = {}
        for layer, items in sorted(by_layer.items()):
            for idx, stage in enumerate(items):
                positions[stage.operator_id] = {"x": start_x + layer * x_gap, "y": start_y + idx * y_gap}
        return positions


class WorkflowJsonAssembler:
    def __init__(self):
        self.operator_builder = WorkflowOperatorBuilder()
        self.link_builder = WorkflowLinkBuilder()
        self.layout_builder = WorkflowLayoutBuilder()

    def assemble(self, stages: list[StageInvocation]) -> dict[str, Any]:
        operators = [
            self.operator_builder.build(stage, view_result=(i == len(stages) - 1))
            for i, stage in enumerate(stages)
        ]
        return {
            "operators": operators,
            "operatorPositions": self.layout_builder.build_positions(stages),
            "links": self.link_builder.build_links(stages),
            "commentBoxes": [],
            "settings": {"dataTransferBatchSize": 400},
        }


class WorkflowTranslator:
    def __init__(self):
        self.parser = SourceModuleParser()
        self.imports = ImportCollector()
        self.functions = FunctionIndex()
        self.call_graph = CallGraphBuilder()
        self.recursion = RecursiveAnalyzer()
        self.facts = FunctionFactsAnalyzer()
        self.constant_env = ConstantEnvBuilder()
        self.dag_extractor = EntryDagExtractor()
        self.ids = WorkflowIdFactory()
        self.code_emitter = FunctionBodyCompiler()
        self.workflow = WorkflowJsonAssembler()
        self.schema_inferer = SourceSchemaInferer()
        self.table_schema_inferer = TableOutputSchemaInferer()

    def translate(self, source: str, entry: str = "main") -> dict[str, Any]:
        tree = self.parser.parse(source)
        import_nodes = self.imports.collect_nodes(tree)
        functions = self.functions.collect(tree)
        if entry not in functions:
            raise ValueError(f"Entry function {entry!r} not found. Available: {sorted(functions)}")

        graph = self.call_graph.build(functions)
        recursive = self.recursion.detect(graph)
        facts = self.facts.analyze(tree, functions, graph, recursive)
        const_env = self.constant_env.build(functions[entry])
        stages = self.dag_extractor.extract(functions[entry], functions, facts, const_env)
        if not stages:
            raise ValueError(f"Entry function {entry!r} does not contain a detectable pipeline of helper calls.")

        stage_by_id: dict[str, StageInvocation] = {s.stage_id: s for s in stages}
        for stage in stages:
            fn = functions[stage.function_name]
            if len(stage.data_inputs) > 2:
                raise ValueError(f"Current workflow JSON emitter supports at most 2 data inputs per stage, got {len(stage.data_inputs)} for {stage.function_name!r}.")
            stage.operator_type = UdfBaseSelector().workflow_operator_type(stage.family, stage.is_source, len(stage.data_inputs))
            stage.operator_id = self.ids.operator_id(stage.operator_type)
            stage.display_name = " ".join(part.capitalize() for part in stage.function_name.split("_"))
            stage.source_columns = self.schema_inferer.infer(fn, stage)
            if stage.operator_type == "PythonUDFSourceV2" and stage.family == "table" and not stage.source_columns:
                raise ValueError(
                    f"Could not infer output columns for source stage {stage.function_name!r}. "
                    "Make the source schema explicit or use a source that can be sampled at translation time."
                )
            if stage.family == "table":
                input_columns: list[dict[str, str]] = []
                for di in stage.data_inputs:
                    upstream = stage_by_id[di.upstream_stage_id]
                    input_columns.extend(upstream.output_columns or upstream.source_columns)
                if stage.is_source:
                    stage.output_columns = list(stage.source_columns)
                    stage.retain_input_columns = False
                else:
                    stage.output_columns, stage.retain_input_columns = self.table_schema_inferer.infer(fn, stage, input_columns)
            stage.code = self.code_emitter.emit_class(import_nodes, fn, stage)

        workflow_json = self.workflow.assemble(stages)
        metadata = {
            "entry": entry,
            "stages": [
                {
                    "stage_id": s.stage_id,
                    "function": s.function_name,
                    "operatorID": s.operator_id,
                    "operator_type": s.operator_type,
                    "family": s.family,
                    "base_class": UdfBaseSelector().select(s.family, s.is_source)[1],
                    "method": UdfBaseSelector().select(s.family, s.is_source)[2],
                    "is_source": s.is_source,
                    "is_sink": s.is_sink,
                    "recursive": s.recursive,
                    "has_loop": s.has_loop,
                    "assigned_to": s.assigned_to,
                    "data_inputs": [
                        {
                            "param_name": di.param_name,
                            "upstream_stage_id": di.upstream_stage_id,
                            "upstream_var": di.upstream_var,
                            "input_port": di.input_port,
                        }
                        for di in s.data_inputs
                    ],
                    "ui_parameters": [
                        {
                            "name": p.name,
                            "attr_type": p.attr_type,
                            "default": ast.unparse(p.default_ast) if p.default_ast is not None else None,
                        }
                        for p in s.ui_params
                    ],
                    "source_columns": s.source_columns,
                    "output_columns": s.output_columns,
                    "retain_input_columns": s.retain_input_columns,
                }
                for s in stages
            ],
        }
        return {"workflow_json": workflow_json, "metadata": metadata}


def translate_to_texera_workflow_json(source: str, entry: str = "main") -> dict[str, Any]:
    return WorkflowTranslator().translate(source, entry=entry)


def write_translation(result: dict[str, Any], out_prefix: str | Path) -> dict[str, str]:
    out_prefix = Path(out_prefix)
    base = out_prefix.with_suffix("")
    workflow_path = base.parent / f"{base.name}.workflow.json"
    meta_path = base.parent / f"{base.name}.workflow.meta.json"
    workflow_path.write_text(json.dumps(result["workflow_json"], indent=2) + "\n", encoding="utf-8")
    meta_path.write_text(json.dumps(result["metadata"], indent=2) + "\n", encoding="utf-8")
    return {"workflow_json": str(workflow_path), "workflow_metadata": str(meta_path)}

# =============================
# Native-operator knowledge base
# =============================

@dataclass
class NativeOperatorMatch:
    operator_type: str
    properties: dict[str, Any]
    display_name: str | None = None
    confidence: float = 1.0
    output_columns: list[dict[str, str]] | None = None
    retain_input_columns: bool | None = None
    notes: list[str] = field(default_factory=list)


class NativeOperatorCatalog:
    def __init__(self, catalog_json_path: str | Path | None = None):
        self.catalog_json_path = Path(catalog_json_path) if catalog_json_path else None
        self.templates: dict[str, dict[str, Any]] = {}
        self._load_default_if_available()

    def _load_default_if_available(self):
        if self.catalog_json_path is None:
            default = Path('/mnt/data/Untitled workflow (11).json')
            if default.exists():
                self.catalog_json_path = default
        if self.catalog_json_path and self.catalog_json_path.exists():
            self.load(self.catalog_json_path)

    def load(self, path: str | Path):
        path = Path(path)
        obj = json.loads(path.read_text(encoding='utf-8'))
        templates: dict[str, dict[str, Any]] = {}
        for op in obj.get('operators', []):
            op_type = op.get('operatorType')
            if op_type and op_type not in templates:
                templates[op_type] = copy.deepcopy(op)
        self.templates = templates

    def has(self, operator_type: str) -> bool:
        return operator_type in self.templates

    def template(self, operator_type: str) -> dict[str, Any]:
        if operator_type not in self.templates:
            raise KeyError(f'Operator type {operator_type!r} not found in native operator catalog')
        return copy.deepcopy(self.templates[operator_type])


class ImportAliasIndex:
    def build(self, tree: ast.Module) -> dict[str, str]:
        aliases: dict[str, str] = {}
        for node in tree.body:
            if isinstance(node, ast.Import):
                for alias in node.names:
                    aliases[alias.asname or alias.name.split('.')[0]] = alias.name
            elif isinstance(node, ast.ImportFrom):
                mod = node.module or ''
                for alias in node.names:
                    aliases[alias.asname or alias.name] = f"{mod}.{alias.name}" if mod else alias.name
        return aliases


class QualifiedNameResolver:
    def __init__(self, aliases: dict[str, str]):
        self.aliases = aliases

    def resolve(self, node: ast.AST | None) -> str | None:
        if node is None:
            return None
        if isinstance(node, ast.Name):
            return self.aliases.get(node.id, node.id)
        if isinstance(node, ast.Attribute):
            base = self.resolve(node.value)
            if base is None:
                return None
            return f"{base}.{node.attr}"
        return None


@dataclass
class CallSignature:
    qualified_names: tuple[str, ...]
    positional_params: tuple[str, ...] = ()
    keyword_aliases: dict[str, str] = field(default_factory=dict)


class CallArgumentBinder:
    def bind(self, call: ast.Call, sig: CallSignature) -> dict[str, ast.AST]:
        bound: dict[str, ast.AST] = {}
        for idx, arg in enumerate(call.args):
            if idx < len(sig.positional_params):
                bound[sig.positional_params[idx]] = arg
        for kw in call.keywords:
            if kw.arg is None:
                continue
            canonical = sig.keyword_aliases.get(kw.arg, kw.arg)
            bound[canonical] = kw.value
        return bound


class StageSemanticFacts:
    def __init__(self, fn: ast.FunctionDef, resolver: QualifiedNameResolver):
        self.fn = fn
        self.resolver = resolver
        self.calls: list[ast.Call] = []
        self.assignments: list[ast.Assign] = []
        self.returns: list[ast.Return] = []
        self.name_to_value: dict[str, ast.AST] = {}
        self._scan()

    def _scan(self):
        for stmt in self.fn.body:
            if isinstance(stmt, ast.Assign):
                self.assignments.append(stmt)
                if len(stmt.targets) == 1 and isinstance(stmt.targets[0], ast.Name):
                    self.name_to_value[stmt.targets[0].id] = stmt.value
            elif isinstance(stmt, ast.Return):
                self.returns.append(stmt)
            for node in ast.walk(stmt):
                if isinstance(node, ast.Call):
                    self.calls.append(node)

    def call_qname(self, call: ast.Call) -> str | None:
        return self.resolver.resolve(call.func)

    def all_call_names(self) -> list[str]:
        out = []
        for call in self.calls:
            name = self.call_qname(call)
            if name:
                out.append(name)
        return out

    def final_return_value(self) -> ast.AST | None:
        for stmt in reversed(self.fn.body):
            if isinstance(stmt, ast.Return):
                return stmt.value
        return None


class NativeMatchContext:
    def __init__(self, stage: StageInvocation, fn: ast.FunctionDef, facts: StageSemanticFacts,
                 input_columns: list[dict[str, str]], const_env: dict[str, ast.AST]):
        self.stage = stage
        self.fn = fn
        self.facts = facts
        self.input_columns = input_columns
        self.const_env = const_env

    @property
    def input_column_names(self) -> list[str]:
        return [c.get('attributeName', '') for c in self.input_columns if c.get('attributeName')]

    def resolve_const(self, node: ast.AST | None) -> ast.AST | None:
        if isinstance(node, ast.Name) and node.id in self.const_env:
            return self.const_env[node.id]
        return node

    def const_value(self, node: ast.AST | None) -> Any:
        node = self.resolve_const(node)
        if isinstance(node, ast.Constant):
            return node.value
        return None

    def subscript_column(self, node: ast.AST | None) -> str | None:
        if isinstance(node, ast.Subscript):
            sl = node.slice
            if isinstance(sl, ast.Constant) and isinstance(sl.value, str):
                return sl.value
        return None


class NativeMatcher:
    def match(self, ctx: NativeMatchContext) -> NativeOperatorMatch | None:
        raise NotImplementedError


class PandasReadCsvMatcher(NativeMatcher):
    SIG = CallSignature(
        qualified_names=('pandas.read_csv', 'pd.read_csv'),
        positional_params=('filepath_or_buffer',),
        keyword_aliases={'filepath_or_buffer': 'filepath_or_buffer', 'sep': 'sep', 'delimiter': 'sep', 'header': 'header'},
    )

    def __init__(self):
        self.binder = CallArgumentBinder()

    def match(self, ctx: NativeMatchContext) -> NativeOperatorMatch | None:
        if not ctx.stage.is_source:
            return None
        for call in ctx.facts.calls:
            qn = ctx.facts.call_qname(call)
            if qn not in self.SIG.qualified_names:
                continue
            args = self.binder.bind(call, self.SIG)
            file_node = ctx.resolve_const(args.get('filepath_or_buffer'))
            file_value = ctx.const_value(file_node)
            if isinstance(file_value, str) and file_value:
                file_name = file_value
            elif file_node is not None:
                file_name = ast.unparse(file_node)
            else:
                file_name = ''
            sep = ctx.const_value(args.get('sep')) or ','
            header = args.get('header')
            has_header = ctx.const_value(header)
            if has_header is None:
                has_header = True
            props = {
                'fileEncoding': 'UTF_8',
                'customDelimiter': sep,
                'hasHeader': bool(has_header),
                'fileName': file_name,
            }
            return NativeOperatorMatch(
                operator_type='CSVFileScan',
                properties=props,
                display_name='CSV File Scan',
                confidence=0.99,
                output_columns=ctx.stage.source_columns or None,
                retain_input_columns=False,
                notes=['Matched pandas.read_csv to CSVFileScan'],
            )
        return None


class HistPlotMatcher(NativeMatcher):
    def match(self, ctx: NativeMatchContext) -> NativeOperatorMatch | None:
        for call in ctx.facts.calls:
            qn = ctx.facts.call_qname(call)
            col = None
            if qn in ('matplotlib.pyplot.hist', 'plt.hist') and call.args:
                col = ctx.subscript_column(call.args[0])
            elif qn in ('seaborn.histplot', 'sns.histplot'):
                for kw in call.keywords:
                    if kw.arg == 'x':
                        if isinstance(kw.value, ast.Constant) and isinstance(kw.value.value, str):
                            col = kw.value.value
                        else:
                            col = ctx.subscript_column(kw.value)
            elif qn in ('plotly.express.histogram', 'px.histogram'):
                for kw in call.keywords:
                    if kw.arg in ('x', 'y') and isinstance(kw.value, ast.Constant) and isinstance(kw.value.value, str):
                        col = kw.value.value
                        break
            elif qn is None and isinstance(call.func, ast.Attribute) and call.func.attr == 'plot':
                base = call.func.value
                if isinstance(base, ast.Call) and isinstance(base.func, ast.Attribute) and base.func.attr == 'dropna':
                    if call.keywords:
                        for kw in call.keywords:
                            if kw.arg == 'kind' and isinstance(kw.value, ast.Constant) and kw.value.value == 'hist':
                                col = ctx.subscript_column(base.func.value)
                                break
            if col:
                return NativeOperatorMatch(
                    operator_type='Histogram',
                    properties={'value': col},
                    display_name='Histogram',
                    confidence=0.95,
                    output_columns=ctx.input_columns,
                    retain_input_columns=True,
                    notes=[f'Matched histogram on column {col!r}'],
                )
        return None


class ScatterPlotMatcher(NativeMatcher):
    def match(self, ctx: NativeMatchContext) -> NativeOperatorMatch | None:
        for call in ctx.facts.calls:
            qn = ctx.facts.call_qname(call)
            x = y = None
            if qn in ('matplotlib.pyplot.scatter', 'plt.scatter') and len(call.args) >= 2:
                x = ctx.subscript_column(call.args[0])
                y = ctx.subscript_column(call.args[1])
            elif qn in ('seaborn.scatterplot', 'sns.scatterplot'):
                for kw in call.keywords:
                    if kw.arg == 'x':
                        x = kw.value.value if isinstance(kw.value, ast.Constant) else ctx.subscript_column(kw.value)
                    elif kw.arg == 'y':
                        y = kw.value.value if isinstance(kw.value, ast.Constant) else ctx.subscript_column(kw.value)
            elif qn in ('plotly.express.scatter', 'px.scatter'):
                for kw in call.keywords:
                    if kw.arg == 'x' and isinstance(kw.value, ast.Constant):
                        x = kw.value.value
                    elif kw.arg == 'y' and isinstance(kw.value, ast.Constant):
                        y = kw.value.value
            if x and y:
                return NativeOperatorMatch(
                    operator_type='Scatterplot',
                    properties={'xColumn': x, 'yColumn': y, 'alpha': 1.0, 'xLogScale': False, 'yLogScale': False},
                    display_name='Scatter Plot',
                    confidence=0.95,
                    output_columns=ctx.input_columns,
                    retain_input_columns=True,
                    notes=[f'Matched scatter plot on {x!r}, {y!r}'],
                )
        return None


class PieChartMatcher(NativeMatcher):
    def match(self, ctx: NativeMatchContext) -> NativeOperatorMatch | None:
        for call in ctx.facts.calls:
            qn = ctx.facts.call_qname(call)
            value = name = None
            if qn in ('matplotlib.pyplot.pie', 'plt.pie') and call.args:
                value = ctx.subscript_column(call.args[0])
                for kw in call.keywords:
                    if kw.arg == 'labels':
                        name = ctx.subscript_column(kw.value)
            elif qn in ('plotly.express.pie', 'px.pie'):
                for kw in call.keywords:
                    if kw.arg == 'values' and isinstance(kw.value, ast.Constant):
                        value = kw.value.value
                    elif kw.arg == 'names' and isinstance(kw.value, ast.Constant):
                        name = kw.value.value
            if value and name:
                return NativeOperatorMatch(
                    operator_type='PieChart',
                    properties={'value': value, 'name': name},
                    display_name='Pie Chart',
                    confidence=0.93,
                    output_columns=ctx.input_columns,
                    retain_input_columns=True,
                )
        return None


class WordCloudMatcher(NativeMatcher):
    def match(self, ctx: NativeMatchContext) -> NativeOperatorMatch | None:
        # Look for WordCloud().generate(_from_text)?(table['col'])
        for call in ctx.facts.calls:
            qn = ctx.facts.call_qname(call)
            if qn not in ('wordcloud.WordCloud', 'WordCloud'):
                continue
            generated_col = None
            top_n = 200
            for kw in call.keywords:
                if kw.arg in ('max_words', 'top_n'):
                    v = ctx.const_value(kw.value)
                    if isinstance(v, int):
                        top_n = v
            # search users of constructor result
        for node in ast.walk(ctx.fn):
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr in ('generate', 'generate_from_text') and node.args:
                generated_col = ctx.subscript_column(node.args[0])
                if generated_col:
                    return NativeOperatorMatch(
                        operator_type='WordCloud',
                        properties={'topN': top_n, 'textColumn': generated_col},
                        display_name='Word Cloud',
                        confidence=0.92,
                        output_columns=ctx.input_columns,
                        retain_input_columns=True,
                    )
        return None


class NativeOperatorKnowledgeBase:
    def __init__(self):
        self.matchers: list[NativeMatcher] = [
            PandasReadCsvMatcher(),
            HistPlotMatcher(),
            ScatterPlotMatcher(),
            PieChartMatcher(),
            WordCloudMatcher(),
        ]

    def match(self, ctx: NativeMatchContext) -> NativeOperatorMatch | None:
        best: NativeOperatorMatch | None = None
        for matcher in self.matchers:
            match = matcher.match(ctx)
            if match is not None and (best is None or match.confidence > best.confidence):
                best = match
        return best


class NativeStagePlanner:
    def __init__(self, catalog: NativeOperatorCatalog | None = None):
        self.catalog = catalog or NativeOperatorCatalog()
        self.aliases = ImportAliasIndex()
        self.knowledge = NativeOperatorKnowledgeBase()

    def plan(self, tree: ast.Module, fn: ast.FunctionDef, stage: StageInvocation,
             input_columns: list[dict[str, str]], const_env: dict[str, ast.AST]) -> NativeOperatorMatch | None:
        alias_map = self.aliases.build(tree)
        resolver = QualifiedNameResolver(alias_map)
        facts = StageSemanticFacts(fn, resolver)
        ctx = NativeMatchContext(stage, fn, facts, input_columns, const_env)
        match = self.knowledge.match(ctx)
        if match is None:
            return None
        if not self.catalog.has(match.operator_type):
            return None
        return match


# =============================
# Override workflow builders with native support
# =============================

class WorkflowOperatorBuilder:
    def __init__(self, native_catalog: NativeOperatorCatalog | None = None):
        self.native_catalog = native_catalog or NativeOperatorCatalog()

    def _operator_properties(self, stage: StageInvocation) -> dict[str, Any]:
        if getattr(stage, 'native_operator_type', None):
            props = copy.deepcopy(getattr(stage, 'native_operator_properties', {}) or {})
            return props
        props: dict[str, Any] = {
            'code': stage.code,
            'workers': 1,
        }
        if stage.operator_type == 'PythonUDFSourceV2':
            if stage.source_columns:
                props['columns'] = stage.source_columns
        else:
            props['retainInputColumns'] = stage.retain_input_columns
            if stage.output_columns:
                props['outputColumns'] = stage.output_columns
        return props

    def _input_ports(self, stage: StageInvocation) -> list[dict[str, Any]]:
        if getattr(stage, 'native_operator_type', None):
            template = self.native_catalog.template(stage.native_operator_type)
            ports = copy.deepcopy(template.get('inputPorts', []))
            count = len(stage.data_inputs)
            if count and len(ports) >= count:
                for i in range(count):
                    if i < len(ports):
                        ports[i]['portID'] = f'input-{i}'
                        if count > 1 and i < len(stage.data_inputs):
                            ports[i]['displayName'] = stage.data_inputs[i].param_name
                            ports[i]['dependencies'] = [] if i == 0 else [{'id': i - 1, 'internal': False}]
                return ports[:count]
            return ports
        if stage.is_source:
            return []
        count = max(1, len(stage.data_inputs))
        ports = []
        for i in range(count):
            display_name = ''
            if count > 1 and i < len(stage.data_inputs):
                display_name = stage.data_inputs[i].param_name
            dependencies: list[dict[str, Any]] = []
            if count > 1 and i > 0:
                dependencies = [{'id': i - 1, 'internal': False}]
            ports.append({
                'portID': f'input-{i}',
                'displayName': display_name,
                'allowMultiInputs': True,
                'isDynamicPort': False,
                'dependencies': dependencies,
            })
        return ports

    def _dynamic_ports_flags(self, stage: StageInvocation) -> tuple[bool, bool]:
        if getattr(stage, 'native_operator_type', None):
            template = self.native_catalog.template(stage.native_operator_type)
            return bool(template.get('dynamicInputPorts', False)), bool(template.get('dynamicOutputPorts', False))
        if stage.is_source:
            return False, False
        if stage.operator_type == 'DualInputPortsPythonUDFV2':
            return False, False
        return True, True

    def build(self, stage: StageInvocation, view_result: bool = False) -> dict[str, Any]:
        if getattr(stage, 'native_operator_type', None):
            template = self.native_catalog.template(stage.native_operator_type)
            template['operatorID'] = stage.operator_id
            template['operatorType'] = stage.native_operator_type
            template['operatorVersion'] = 'N/A'
            template['operatorProperties'] = self._operator_properties(stage)
            template['inputPorts'] = self._input_ports(stage)
            template['outputPorts'] = copy.deepcopy(template.get('outputPorts', [
                {'portID': 'output-0', 'displayName': '', 'allowMultiInputs': False, 'isDynamicPort': False}
            ]))
            template['customDisplayName'] = stage.display_name
            if view_result:
                template['viewResult'] = True
            return template

        dynamic_input_ports, dynamic_output_ports = self._dynamic_ports_flags(stage)
        operator = {
            'operatorID': stage.operator_id,
            'operatorType': stage.operator_type,
            'operatorVersion': 'N/A',
            'operatorProperties': self._operator_properties(stage),
            'inputPorts': self._input_ports(stage),
            'outputPorts': [
                {'portID': 'output-0', 'displayName': '', 'allowMultiInputs': False, 'isDynamicPort': False}
            ],
            'showAdvanced': False,
            'isDisabled': False,
            'customDisplayName': stage.display_name,
            'dynamicInputPorts': dynamic_input_ports,
            'dynamicOutputPorts': dynamic_output_ports,
        }
        if view_result:
            operator['viewResult'] = True
        return operator


class WorkflowJsonAssembler:
    def __init__(self, native_catalog: NativeOperatorCatalog | None = None):
        self.operator_builder = WorkflowOperatorBuilder(native_catalog=native_catalog)
        self.link_builder = WorkflowLinkBuilder()
        self.layout_builder = WorkflowLayoutBuilder()

    def assemble(self, stages: list[StageInvocation]) -> dict[str, Any]:
        operators = [
            self.operator_builder.build(stage, view_result=(i == len(stages) - 1))
            for i, stage in enumerate(stages)
        ]
        return {
            'operators': operators,
            'operatorPositions': self.layout_builder.build_positions(stages),
            'links': self.link_builder.build_links(stages),
            'commentBoxes': [],
            'settings': {'dataTransferBatchSize': 400},
        }


class WorkflowTranslator:
    def __init__(self, catalog_json_path: str | Path | None = None):
        self.parser = SourceModuleParser()
        self.imports = ImportCollector()
        self.functions = FunctionIndex()
        self.call_graph = CallGraphBuilder()
        self.recursion = RecursiveAnalyzer()
        self.facts = FunctionFactsAnalyzer()
        self.constant_env = ConstantEnvBuilder()
        self.dag_extractor = EntryDagExtractor()
        self.ids = WorkflowIdFactory()
        self.code_emitter = FunctionBodyCompiler()
        self.catalog = NativeOperatorCatalog(catalog_json_path)
        self.native_planner = NativeStagePlanner(self.catalog)
        self.workflow = WorkflowJsonAssembler(native_catalog=self.catalog)
        self.schema_inferer = SourceSchemaInferer()
        self.table_schema_inferer = TableOutputSchemaInferer()

    def translate(self, source: str, entry: str = 'main') -> dict[str, Any]:
        tree = self.parser.parse(source)
        import_nodes = self.imports.collect_nodes(tree)
        functions = self.functions.collect(tree)
        if entry not in functions:
            raise ValueError(f"Entry function {entry!r} not found. Available: {sorted(functions)}")

        graph = self.call_graph.build(functions)
        recursive = self.recursion.detect(graph)
        facts = self.facts.analyze(tree, functions, graph, recursive)
        const_env = self.constant_env.build(functions[entry])
        stages = self.dag_extractor.extract(functions[entry], functions, facts, const_env)
        if not stages:
            raise ValueError(f"Entry function {entry!r} does not contain a detectable pipeline of helper calls.")

        stage_by_id: dict[str, StageInvocation] = {s.stage_id: s for s in stages}
        selector = UdfBaseSelector()
        for stage in stages:
            fn = functions[stage.function_name]
            if len(stage.data_inputs) > 2:
                raise ValueError(f"Current workflow JSON emitter supports at most 2 data inputs per stage, got {len(stage.data_inputs)} for {stage.function_name!r}.")
            stage.display_name = ' '.join(part.capitalize() for part in stage.function_name.split('_'))
            # default UDF path
            stage.operator_type = selector.workflow_operator_type(stage.family, stage.is_source, len(stage.data_inputs))
            stage.operator_id = self.ids.operator_id(stage.operator_type)
            stage.source_columns = self.schema_inferer.infer(fn, stage)
            if stage.family == 'table':
                input_columns: list[dict[str, str]] = []
                for di in stage.data_inputs:
                    upstream = stage_by_id[di.upstream_stage_id]
                    input_columns.extend(getattr(upstream, 'output_columns', []) or getattr(upstream, 'source_columns', []))
                if stage.is_source:
                    stage.output_columns = list(stage.source_columns)
                    stage.retain_input_columns = False
                else:
                    stage.output_columns, stage.retain_input_columns = self.table_schema_inferer.infer(fn, stage, input_columns)
            else:
                input_columns = []

            native_match = self.native_planner.plan(tree, fn, stage, input_columns, const_env)
            if native_match is not None:
                stage.native_operator_type = native_match.operator_type
                stage.native_operator_properties = native_match.properties
                stage.native_notes = native_match.notes
                stage.operator_type = native_match.operator_type
                stage.operator_id = self.ids.operator_id(native_match.operator_type)
                if native_match.display_name:
                    stage.display_name = native_match.display_name if stage.is_source else stage.display_name
                if native_match.output_columns is not None:
                    stage.output_columns = native_match.output_columns
                    if stage.is_source:
                        stage.source_columns = native_match.output_columns
                if native_match.retain_input_columns is not None:
                    stage.retain_input_columns = native_match.retain_input_columns
                stage.code = ''
            else:
                if stage.operator_type == 'PythonUDFSourceV2' and stage.family == 'table' and not stage.source_columns:
                    raise ValueError(
                        f"Could not infer output columns for source stage {stage.function_name!r}. "
                        'Make the source schema explicit or use a source that can be sampled at translation time.'
                    )
                stage.code = self.code_emitter.emit_class(import_nodes, fn, stage)

        workflow_json = self.workflow.assemble(stages)
        metadata = {
            'entry': entry,
            'catalog_path': str(self.catalog.catalog_json_path) if self.catalog.catalog_json_path else None,
            'stages': [
                {
                    'stage_id': s.stage_id,
                    'function': s.function_name,
                    'operatorID': s.operator_id,
                    'operator_type': s.operator_type,
                    'native_operator_type': getattr(s, 'native_operator_type', None),
                    'family': s.family,
                    'base_class': None if getattr(s, 'native_operator_type', None) else selector.select(s.family, s.is_source)[1],
                    'method': None if getattr(s, 'native_operator_type', None) else selector.select(s.family, s.is_source)[2],
                    'is_source': s.is_source,
                    'is_sink': s.is_sink,
                    'recursive': s.recursive,
                    'has_loop': s.has_loop,
                    'assigned_to': s.assigned_to,
                    'data_inputs': [
                        {
                            'param_name': di.param_name,
                            'upstream_stage_id': di.upstream_stage_id,
                            'upstream_var': di.upstream_var,
                            'input_port': di.input_port,
                        }
                        for di in s.data_inputs
                    ],
                    'ui_parameters': [
                        {
                            'name': p.name,
                            'attr_type': p.attr_type,
                            'default': ast.unparse(p.default_ast) if p.default_ast is not None else None,
                        }
                        for p in s.ui_params
                    ],
                    'source_columns': s.source_columns,
                    'output_columns': s.output_columns,
                    'retain_input_columns': s.retain_input_columns,
                    'native_notes': getattr(s, 'native_notes', []),
                }
                for s in stages
            ],
        }
        return {'workflow_json': workflow_json, 'metadata': metadata}


def translate_to_texera_workflow_json(source: str, entry: str = 'main', catalog_json_path: str | Path | None = None) -> dict[str, Any]:
    return WorkflowTranslator(catalog_json_path=catalog_json_path).translate(source, entry=entry)

# =============================
# Native matcher extensions added in v15
# =============================

class TimeSeriesPlotMatcher(NativeMatcher):
    """Match simple matplotlib/seaborn/plotly line plots over a column pair.

    Conservative mapping:
      plt.plot(table['date'], table['close']) -> TimeSeriesPlot
    We avoid LineChart here because the catalog template exposes a less explicit
    nested `lines` shape, while TimeSeriesPlot has a stable flat property schema.
    """

    def match(self, ctx: NativeMatchContext) -> NativeOperatorMatch | None:
        for call in ctx.facts.calls:
            qn = ctx.facts.call_qname(call)
            x = y = None

            if qn in ('matplotlib.pyplot.plot', 'plt.plot') and len(call.args) >= 2:
                x = ctx.subscript_column(call.args[0])
                y = ctx.subscript_column(call.args[1])
            elif qn in ('seaborn.lineplot', 'sns.lineplot'):
                for kw in call.keywords:
                    if kw.arg == 'x':
                        x = kw.value.value if isinstance(kw.value, ast.Constant) and isinstance(kw.value.value, str) else ctx.subscript_column(kw.value)
                    elif kw.arg == 'y':
                        y = kw.value.value if isinstance(kw.value, ast.Constant) and isinstance(kw.value.value, str) else ctx.subscript_column(kw.value)
            elif qn in ('plotly.express.line', 'px.line'):
                for kw in call.keywords:
                    if kw.arg == 'x' and isinstance(kw.value, ast.Constant) and isinstance(kw.value.value, str):
                        x = kw.value.value
                    elif kw.arg == 'y' and isinstance(kw.value, ast.Constant) and isinstance(kw.value.value, str):
                        y = kw.value.value

            if x and y:
                return NativeOperatorMatch(
                    operator_type='TimeSeriesPlot',
                    properties={
                        'timeColumn': x,
                        'valueColumn': y,
                        'line': y,
                        'facetColumn': 'No Selection',
                        'categoryColumn': 'No Selection',
                        'slider': False,
                    },
                    display_name='Time Series Plot',
                    confidence=0.96,
                    output_columns=ctx.input_columns,
                    retain_input_columns=True,
                    notes=[f'Matched line/time-series plot on x={x!r}, y={y!r}'],
                )
        return None


class NativeOperatorKnowledgeBase:
    def __init__(self):
        self.matchers: list[NativeMatcher] = [
            PandasReadCsvMatcher(),
            TimeSeriesPlotMatcher(),
            HistPlotMatcher(),
            ScatterPlotMatcher(),
            PieChartMatcher(),
            WordCloudMatcher(),
        ]

    def match(self, ctx: NativeMatchContext) -> NativeOperatorMatch | None:
        best: NativeOperatorMatch | None = None
        for matcher in self.matchers:
            match = matcher.match(ctx)
            if match is not None and (best is None or match.confidence > best.confidence):
                best = match
        return best
if __name__ == "__main__":
    result = translate_to_texera_workflow_json(sample, entry="main", catalog_json_path="catalog.json")
    paths = write_translation(result, "demo_texera_workflow_vx2")
    print(json.dumps(paths, indent=2))

{
  "workflow_json": "demo_texera_workflow_vx2.workflow.json",
  "workflow_metadata": "demo_texera_workflow_vx2.workflow.meta.json"
}


{
  "workflow_json": "demo_texera_workflow_vx2.workflow.json",
  "workflow_metadata": "demo_texera_workflow_vx2.workflow.meta.json"
}


In [7]:
sample = '''
import pandas as pd
import matplotlib.pyplot as plt

def extract_data(url: str) -> pd.DataFrame:
    df = pd.read_csv(url)
    print("Step 1: Extract")
    print(df.head())
    return df

def select_columns(df: pd.DataFrame) -> pd.DataFrame:
    selected = df[["date", "open", "high", "low", "close", "volume"]].copy()
    print("\\nStep 2: Select columns")
    print(selected.head())
    return selected

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.dropna().copy()
    print("\\nStep 3: Clean data")
    print(cleaned.head())
    return cleaned

def transform_data(df: pd.DataFrame) -> pd.DataFrame:
    transformed = df.copy()
    transformed["date"] = pd.to_datetime(transformed["date"], errors="coerce")
    transformed = transformed.dropna(subset=["date"]).copy()
    transformed = transformed.sort_values("date")
    transformed["daily_return"] = transformed["close"].pct_change() * 100
    transformed["ma_30"] = transformed["close"].rolling(window=30).mean()
    print("\\nStep 4: Transform data")
    print(transformed.head())
    return transformed

def load_data(df: pd.DataFrame, output_file: str) -> None:
    df.to_csv(output_file, index=False)
    print(f"\\nStep 5: Load data")
    print(f"Saved cleaned data to: {output_file}")

def plot_data(df: pd.DataFrame) -> None:
    plt.figure(figsize=(10, 5))
    plt.plot(df["date"], df["close"])
    plt.show()

def main():
    url = "https://raw.githubusercontent.com/vega/vega-datasets/main/data/sp500-2000.csv"
    output_file = "clean_sp500_2000.csv"

    raw_df = extract_data(url)
    selected_df = select_columns(raw_df)
    cleaned_df = clean_data(selected_df)
    transformed_df = transform_data(cleaned_df)
    load_data(transformed_df, output_file)
    plot_data(transformed_df)'''